In [19]:
from datetime import date,timedelta
import httpx

today = date.today() - timedelta(days=1)
data = {
    "query": f"publication-date={today.strftime('%Y%m%d')}",
    "fields": [
        "publication-date",
        "notice-title",
        "procedure-type",
        "contract-nature",
        # TODO fix country
        # "country",
        "tender-value",
        "tender-value-cur",
        "classification-cpv",
        "organisation-contact-point-tenderer",
        "document-url-lot"
    ],
    "page": 1,
    "limit": 10,
    "scope": "ACTIVE",
    "checkQuerySyntax": False,
    "paginationMode": "ITERATION"
}

r = httpx.post('https://api.ted.europa.eu/v3/notices/search', json=data)
r.json()

{'notices': [{'organisation-contact-point-tenderer': ['Jiri Stanek',
    'Johan Håkansson'],
   'procedure-type': 'open',
   'classification-cpv': ['77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77310000',
    '77311000',
    '77314000',
    '77300000',
    '77300000',
    '77300000',
    '77300000',
    '77300000',
    '77300000'],
   'publication-number': '32737-2025',
   'contract-nature': ['services',
    'services',
    'services',
    'services',
    'services',
    'services'],
   'publication-date': '2025-01-17+01:00',
   'links': {'xml': {'MUL': 'https://ted.europa.eu/en/notice/32737-2025/xml'},
    'pdf': {'BUL': 'https://ted.europa.eu/bg/notice/32737-2025/pdf',
     'SPA': 'https://ted.europa.eu/es/notice/32737-2025/pdf',
     'CES': 'https://ted.europa.eu/cs/notice/32737-2025/pdf',
 

In [1]:
import os

import httpx
from dotenv import load_dotenv
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session

load_dotenv(".env")


client_id = os.environ["PUBPROC_CLIENT_ID"]
client_secret = os.environ["PUBPROC_CLIENT_SECRET"]

url = "https://public.int.fedservices.be/api/oauth2/token"

client = BackendApplicationClient(client_id=client_id)
oauth = OAuth2Session(client=client)

token = oauth.fetch_token(
    token_url=url,
    client_id=client_id,
    client_secret=client_secret,
    include_client_id=True,
)

print(token["access_token"], token["expires_at"])

/Users/mehmet/projects/proclogic_api/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


hHXocK8tcdSAThJSqdxt6Fvwl2QrMzfZGlw80IemQ8e5NGKVDSalNf 1739385023.443857


In [14]:
from datetime import date, timedelta

def get_nearest_business_day(date_obj: date = None) -> date:
    if date_obj is None:
        date_obj = date.today()

    if date_obj.weekday() == 5:  # Saturday
        return date_obj - timedelta(days=1)
    elif date_obj.weekday() == 6:  # Sunday
        return date_obj - timedelta(days=2)
    return date_obj

latest_business_day = get_nearest_business_day()

data = {
    # TODO: add cpv based on sector in query
    # TODO: implement batch adding to sql server
    "dispatch-date-from": "2025-01-01",
    "page": 1,
    "pageSize": 100,
}

headers = {
    "Authorization": f"Bearer {token['access_token']}",
    "BelGov-Trace-Id": "2ce83af9-d524-43a6-8d1c-b19dff051aed",
}
r = httpx.get('https://public.int.fedservices.be/api/eProcurementSea/v1/search/publications', params=data, headers=headers)

publications = r.json()["publications"]
print("total count", r.json()["totalCount"])

if r.status_code == 200:
    totalCount = int(r.json()["totalCount"])
    pages = totalCount / 100 if totalCount % 100 == 0 else int(totalCount / 100) + 1

    if pages > 1:
        for i in range(2, pages + 1):
            data["page"] = i
            print("fetching page", i)
            r = httpx.get(
                "https://public.int.fedservices.be/api/eProcurementSea/v1/search/publications",
                params=data,
                headers=headers,
            )
            publications.extend(r.json()["publications"])

publications

total count 117
fetching page 2


[{'cpvAdditionalCodes': [{'code': '45442100-8',
    'descriptions': [{'language': 'EN', 'text': 'Painting work'},
     {'language': 'NL', 'text': 'Schilderwerk'},
     {'language': 'FR', 'text': 'Travaux de peinture'},
     {'language': 'DE', 'text': 'Anstricharbeiten'}]},
   {'code': '45421132-8',
    'descriptions': [{'language': 'EN', 'text': 'Installation of windows'},
     {'language': 'DE', 'text': 'Einbau von Fenstern'},
     {'language': 'FR', 'text': 'Pose de fenêtres'},
     {'language': 'NL', 'text': 'Plaatsen van ramen'}]}],
  'cpvMainCode': {'code': '45421132-8',
   'descriptions': [{'language': 'EN', 'text': 'Installation of windows'},
    {'language': 'DE', 'text': 'Einbau von Fenstern'},
    {'language': 'FR', 'text': 'Pose de fenêtres'},
    {'language': 'NL', 'text': 'Plaatsen van ramen'}]},
  'dispatchDate': '2025-02-12',
  'dossier': {'accreditations': {},
   'descriptions': [{'language': 'NL', 'text': 'MVL006 ramen'}],
   'enterpriseCategories': [],
   'legalBasis'

In [5]:
headers = {
    'Authorization': f'Bearer {token["access_token"]}',
    'BelGov-Trace-Id': '2ce83af9-d524-43a6-8d1c-b19dff051aed'
}

r = httpx.get(
    "https://public.int.fedservices.be/api/eProcurementDos/v1/publication-workspaces/9e7bd1fc-4dbb-4883-ae1c-492000631154",
    headers=headers,
)

r.json()

{'id': '9e7bd1fc-4dbb-4883-ae1c-492000631154',
 'organisationId': 3361,
 'organisationNames': [{'text': 'Mira nv', 'language': 'NL'}],
 'noticeSubType': '17',
 'noticeType': 'cn-general',
 'formType': 'COMPETITION',
 'versions': [{'id': 'f7fd3c5d-de1a-4346-977b-f786fe500abf',
   'dispatchDate': '2024-12-17',
   'submissionDeadline': '2025-01-22T15:40:00',
   'activationDate': '2024-12-17',
   'createdAt': '2024-12-17T14:52:32.852193',
   'sentAt': '2024-12-17T15:46:45.025919',
   'publicationPolicy': 'BDA_ONLY',
   'version': 6,
   'status': 'PUBLISHED',
   'notice': {'id': 'dbbdd9db-e63c-4240-99e8-1ff8c43c238f',
    'xmlContent': '<?xml version="1.0" encoding="UTF-8" standalone="yes"?><ContractNotice xmlns:ext="urn:oasis:names:specification:ubl:schema:xsd:CommonExtensionComponents-2" xmlns="urn:oasis:names:specification:ubl:schema:xsd:ContractNotice-2" xmlns:ns7="urn:oasis:names:specification:ubl:schema:xsd:ContractAwardNotice-2" xmlns:cac="urn:oasis:names:specification:ubl:schema:xsd

In [ ]:
import xmltodict

headers = {
    'Authorization': f'Bearer {token["access_token"]}',
    # TODO: generate_uuid
    "BelGov-Trace-Id": "2ce83af9-d524-43a6-8d1c-b19dff051aed",
}
data = {
    # TODO: fix mediatype to html or pdf, not xml
    "Published": "True",
}

r = httpx.get(
    "https://public.int.fedservices.be/api/eProcurementDos/v1/notices/d2969514-e974-46e6-a2d2-641103b6a96a",
    params=data,
    headers=headers,
)
xmltodict.parse(r.text)

{'ContractNotice': {'@xmlns:ext': 'urn:oasis:names:specification:ubl:schema:xsd:CommonExtensionComponents-2',
  '@xmlns': 'urn:oasis:names:specification:ubl:schema:xsd:ContractNotice-2',
  '@xmlns:ns7': 'urn:oasis:names:specification:ubl:schema:xsd:ContractAwardNotice-2',
  '@xmlns:cac': 'urn:oasis:names:specification:ubl:schema:xsd:CommonAggregateComponents-2',
  '@xmlns:cbc': 'urn:oasis:names:specification:ubl:schema:xsd:CommonBasicComponents-2',
  '@xmlns:ns9': 'urn:oasis:names:specification:ubl:schema:xsd:PriorInformationNotice-2',
  '@xmlns:efac': 'http://data.europa.eu/p27/eforms-ubl-extension-aggregate-components/1',
  '@xmlns:efbc': 'http://data.europa.eu/p27/eforms-ubl-extension-basic-components/1',
  '@xmlns:efext': 'http://data.europa.eu/p27/eforms-ubl-extensions/1',
  'ext:UBLExtensions': {'ext:UBLExtension': {'ext:ExtensionContent': {'efext:EformsExtension': {'efbc:TransmissionDate': '2024-12-18+01:00',
      'efbc:TransmissionTime': '00:00:00+01:00',
      'efac:NoticeSub